In [14]:
import os

In [15]:
%pwd

'c:\\Users\\Koushik Sripathi\\Desktop\\Text-Summarizer'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Koushik Sripathi\\Desktop\\Text-Summarizer'

In [16]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [17]:
from src.TextSummarizer.constants import CONFIG_FILEPATH, PARAMS_FILEPATH
from src.TextSummarizer.utils.common import read_yaml, create_directories

In [18]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath=CONFIG_FILEPATH,
        params_filepath=PARAMS_FILEPATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.unzip_dir])
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config

In [19]:
import os
import urllib.request as request
import zipfile
from src.TextSummarizer.logging import logger
from src.TextSummarizer.utils.common import get_size

In [20]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_url,
                filename = self.config.local_data_file
            )
            logger.info(f"File retrieved successfully: {filename} with the header {headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")
            
            
    def extract_zip_file(self):
        """ zip_file_path -> str, Extracts the zip file into the data directory Function returns None"""
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [21]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    logger.exception(e)


[2026-06-11 12:41:38,528: INFO: common: yaml file: config\config.yaml loaded succesfully]
[2026-06-11 12:41:38,529: INFO: common: yaml file: params.yaml loaded succesfully]
[2026-06-11 12:41:38,531: INFO: common: created a directory at: artifacts]
[2026-06-11 12:41:38,533: INFO: common: created a directory at: artifacts/data_ingestion]
[2026-06-11 12:41:40,242: INFO: 2637279188: File retrieved successfully: artifacts/data_ingestion/data.zip with the header Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 33F4:118904:4E56F6:5A15BB:6A2A73ED
Accept-Ranges: bytes
Date: Thu, 11 Jun 2026 08:41:39 GMT
Via: 1.1 varnish
X-Served-By: cache-fjr99002